## 📊 Proceso transformación de información a capa Silver:
#### 📁 Paso 1: Saneamiento de Texto: Elimina caracteres invisibles saltos de línea, tabuladores.
#### 📁 Paso 2: Normalización de Categorías: Convierte la columna CLASE a mayúsculas y quita espacios, asegurando que DV00 y FV00 sean detectados correctamente 
#### 📁 Paso 3:Gestión de Valores Nulos: Identifica los campos vacíos ("") y los convierte en NULL.
#### 📁 Paso 4: Limpieza Numérica Profunda: * Remueve puntos de miles.Cambia comas decimales por puntos. Filtra cualquier carácter que no sea número, punto o signo negativo.
#### 📁 Paso 5: Casteo de Datos (Try-Cast): Convierte las columnas de texto a tipo Double.
#### 📁 Paso 6: Estandarización de Tiempos: Transforma la columna FECHA al formato estándar.
#### 📁 Paso 7:Persistencia Delta: Guarda el resultado final en una tabla Delta Lake en el esquema Silver.


In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace,
    to_date,
    year,
    month,
    when,
    expr,
    upper
)

# ==========================================
# LEER TABLA DESDE BRONZE
# ==========================================
df = spark.table("workspace.bronze.movcomercial")

# ==========================================
# LIMPIAR CARACTERES ESPECIALES Y ESPACIOS
# ==========================================
for column in df.columns:

    df = df.withColumn(
        column,
        trim(
            regexp_replace(
                regexp_replace(col(column), r'[\n\r\t]', ' '),
                r'\s+',
                ' '
            )
        )
    )

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# REEMPLAZAR VACÍOS POR NULL
# ==========================================
for column in df.columns:

    df = df.withColumn(
        column,
        when(col(column) == "", None)
        .otherwise(col(column))
    )

# ==========================================
# COLUMNAS NUMÉRICAS
# ==========================================
numeric_columns = [
    "UNIDAD",
    "CANTIDAD",
    "PARCIAL",
    "PARCIALANT",
    "COSTO",
    "PARLANTIVA",
    "UTILIDAD",
    "RENTABILID"
]

# ==========================================
# LIMPIAR Y CONVERTIR COLUMNAS NUMÉRICAS
# ==========================================
for column in numeric_columns:

    # Eliminar separador de miles "."
    df = df.withColumn(
        column,
        regexp_replace(col(column), r"\.", "")
    )

    # Reemplazar coma decimal "," por "."
    df = df.withColumn(
        column,
        regexp_replace(col(column), ",", ".")
    )

    # Eliminar caracteres no numéricos
    df = df.withColumn(
        column,
        regexp_replace(col(column), r"[^0-9\.-]", "")
    )

    # Convertir a DOUBLE
    df = df.withColumn(
        column,
        expr(f"try_cast({column} as double)")
    )

# ==========================================
# CREAR COLUMNAS MÁS DESCRIPTIVAS
# ==========================================
df = df.withColumn(
    "PRECIO_VENTA",
    col("PARCIAL")
)

df = df.withColumn(
    "PRECIO_ANTES_IVA",
    col("PARCIALANT")
)

df = df.withColumn(
    "RENTABILIDAD",
    col("RENTABILID")
)

# ==========================================
# CONVERTIR FECHA
# ==========================================
df = df.withColumn(
    "FECHA",
    to_date(col("FECHA"), "yyyy-MM-dd HH:mm:ss")
)
# ==========================================
# CREAR COLUMNAS AUXILIARES
# ==========================================
df = df.withColumn(
    "ANIO",
    year(col("FECHA"))
)

df = df.withColumn(
    "MES",
    month(col("FECHA"))
)

# ==========================================
# MOSTRAR RESULTADO
# ==========================================
display(df)
df.printSchema()
print(f"Total registros: {df.count()}")

# ==========================================
# CREAR ESQUEMA SILVER SI NO EXISTE
# ==========================================
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

# ==========================================
# GUARDAR TABLA SILVER
# ==========================================
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.movcomercial")
print("Tabla workspace.silver.movcomercial creada correctamente")

# ==========================================
# VALIDAR TABLA
# ==========================================
df_silver = spark.table("workspace.silver.movcomercial")
display(df_silver)

# ==========================================
# VER DETALLE
# ==========================================
spark.sql("""
DESCRIBE DETAIL workspace.silver.movcomercial
""").display()